# Community Needs Indexing - pull, clean, export

Run this by hand a few times a year. No scheduled jobs, no database - this notebook is the entire pipeline.

**Output:** overwrites `/data/community_needs_index.json`, which `/site/js/main.js` fetches at page load.

**Before running:** fill in the TODOs below with a real WPRDC dataset endpoint (https://data.wprdc.org) or a real Census/ACS table (https://api.census.gov/data.html) for the one indicator you're shipping in v1. Don't guess an endpoint - look it up and paste the real resource ID / table ID in.

In [ ]:
import pandas as pd
import requests
import json
from datetime import date
from pathlib import Path

RAW_DIR = Path('../raw')
DATA_DIR = Path('../data')
RAW_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

today = date.today().isoformat()
print('Pipeline run date:', today)

## 1. Pull

TODO: replace with the real WPRDC resource ID or Census/ACS table + variable for your one v1 indicator.

WPRDC uses a CKAN datastore `sql` endpoint, e.g.:
`https://data.wprdc.org/api/3/action/datastore_search_sql?sql=SELECT * FROM "<resource_id>"`

Census/ACS uses `https://api.census.gov/data/<year>/acs/acs5?get=<variables>&for=county:003&in=state:42`
(Allegheny County FIPS is 003, Pennsylvania state FIPS is 42 - verify against the current ACS table layout before relying on this).

In [ ]:
# TODO: set this to the real endpoint once you've identified the dataset
SOURCE_URL = None  # e.g. 'https://data.wprdc.org/api/3/action/datastore_search_sql?sql=...'

if SOURCE_URL:
    resp = requests.get(SOURCE_URL, timeout=30)
    resp.raise_for_status()
    raw = resp.json()
    # cache the raw pull, dated, for reproducibility
    with open(RAW_DIR / f'community_needs_index_raw_{today}.json', 'w') as f:
        json.dump(raw, f, indent=2)
else:
    print('SOURCE_URL not set yet - fill in a real endpoint before running for real.')

## 2. Clean

TODO: reshape the raw pull into: one row per (year, geography), one value column.
Keep this as a `pandas.DataFrame` with columns `year`, `geography`, `value` before exporting.

In [ ]:
# df = pd.DataFrame(...)  # TODO: build from `raw`
# df = df[['year', 'geography', 'value']].sort_values(['year', 'geography'])

## 3. Export

Pivot to the wide schema the site expects (one object per year, one key per geography), then overwrite `/data/community_needs_index.json`. Keep the `indicator`, `unit`, `source`, and `snapshot_as_of` fields accurate - these are what render as the on-page citation and date stamp.

In [ ]:
# wide = df.pivot(index='year', columns='geography', values='value').reset_index()
# series = wide.to_dict(orient='records')
#
# payload = {
#     'indicator': 'TODO: real indicator name',
#     'geographies': ['Allegheny County', 'Pennsylvania', 'United States'],
#     'unit': 'TODO',
#     'snapshot_as_of': today,
#     'status': 'live',
#     'source': 'TODO: real citation, e.g. WPRDC resource <id>, pulled <today>',
#     'series': series,
#     'notes': ''
# }
#
# with open(DATA_DIR / 'community_needs_index.json', 'w') as f:
#     json.dump(payload, f, indent=2)
# print('Wrote data/community_needs_index.json')